In [0]:
# Make sure everything is where we expect it
base_path = "/Volumes/finalcomp/default/caretocompare/CARE_To_Compare/CARE_To_Compare"

for item in dbutils.fs.ls(base_path):
    print(item.name)

README.md
README.txt
Wind Farm A/
Wind Farm B/
Wind Farm C/


In [0]:
farm_a_path = f"{base_path}/Wind Farm A"

for item in dbutils.fs.ls(farm_a_path):
    print(item.name, f"({item.size / 1e3:.1f} KB)")

datasets/ (0.0 KB)
event_info.csv (1.8 KB)
feature_description.csv (4.1 KB)


In [0]:
import pandas as pd

farm_a_path = f"{base_path}/Wind Farm A"

event_info = pd.read_csv(f"{farm_a_path}/event_info.csv", sep=";")
display(event_info.head(10))
print(f"\nShape: {event_info.shape}")
print(f"Anomaly events: {(event_info['event_label'] == 'anomaly').sum()}")
print(f"Normal events:  {(event_info['event_label'] == 'normal').sum()}")

asset,event_id,event_label,event_start,event_start_id,event_end,event_end_id,event_description
11,68,anomaly,2023-07-28 13:20:00,52063,2023-08-11 13:10:00,54076,Transformer failure
21,22,anomaly,2023-08-12 09:50:00,51888,2023-08-19 10:00:00,52892,Hydraulic group
21,72,anomaly,2023-10-10 08:40:00,52497,2023-10-17 08:40:00,53505,Gearbox failure
0,73,anomaly,2023-06-10 11:40:00,52745,2023-06-17 11:40:00,53753,Hydraulic group
0,0,anomaly,2023-08-06 06:10:00,52436,2023-08-20 06:10:00,54447,Generator bearing failure
0,26,anomaly,2023-10-12 10:20:00,52261,2023-10-19 10:20:00,53269,Hydraulic group
10,40,anomaly,2022-12-26 00:00:00,51363,2023-01-26 13:00:00,55870,Generator bearing failure
10,42,anomaly,2023-09-09 15:50:00,52303,2023-09-16 15:50:00,53309,Hydraulic group
10,10,anomaly,2023-10-11 08:40:00,52611,2023-10-18 08:40:00,53591,Gearbox failure
13,45,anomaly,2023-04-19 18:10:00,52731,2023-04-26 18:10:00,53738,Hydraulic group



Shape: (22, 8)
Anomaly events: 12
Normal events:  10


In [0]:
feature_desc = pd.read_csv(f"{farm_a_path}/feature_description.csv", sep=";")
display(feature_desc.head(20))
print(f"\nTotal features: {len(feature_desc)}")

sensor_name,statistics_type,description,unit,is_angle,is_counter
sensor_0,average,Ambient temperature,�C,false,false
sensor_1,average,Wind absolute direction,�,true,false
sensor_2,average,Wind relative direction,�,true,false
wind_speed_3,"maximum,minimum,average,std_dev",Windspeed,m/s,false,false
wind_speed_4,average,Estimated windspeed,m/s,false,false
sensor_5,"maximum,minimum,std_dev,average",Pitch angle,�,true,false
sensor_6,average,Temperature in the hub controller,�C,false,false
sensor_7,average,Temperature in the top nacelle controller,�C,false,false
sensor_8,average,Temperature in the choke coils on the VCS-section,�C,false,false
sensor_9,average,Temperature on the VCP-board,�C,false,false



Total features: 54


In [0]:
import os

datasets_path = f"{farm_a_path}/datasets"

all_files = os.listdir(datasets_path)
csv_files = [f for f in all_files if f.endswith(".csv")]
print(f"Found {len(csv_files)} dataset files")

dfs = []
for fname in csv_files:
    fpath = os.path.join(datasets_path, fname)
    df = pd.read_csv(fpath, sep=";")
    event_id = fname.replace(".csv", "")
    df["event_id"] = event_id
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(f"Total rows loaded: {len(combined):,}")
print(f"Columns: {list(combined.columns)}")

Found 22 dataset files
Total rows loaded: 1,196,747
Columns: ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id', 'sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'wind_speed_3_avg', 'wind_speed_4_avg', 'wind_speed_3_max', 'wind_speed_3_min', 'wind_speed_3_std', 'sensor_5_avg', 'sensor_5_max', 'sensor_5_min', 'sensor_5_std', 'sensor_6_avg', 'sensor_7_avg', 'sensor_8_avg', 'sensor_9_avg', 'sensor_10_avg', 'sensor_11_avg', 'sensor_12_avg', 'sensor_13_avg', 'sensor_14_avg', 'sensor_15_avg', 'sensor_16_avg', 'sensor_17_avg', 'sensor_18_avg', 'sensor_18_max', 'sensor_18_min', 'sensor_18_std', 'sensor_19_avg', 'sensor_20_avg', 'sensor_21_avg', 'sensor_22_avg', 'sensor_23_avg', 'sensor_24_avg', 'sensor_25_avg', 'sensor_26_avg', 'reactive_power_27_avg', 'reactive_power_27_max', 'reactive_power_27_min', 'reactive_power_27_std', 'reactive_power_28_avg', 'reactive_power_28_max', 'reactive_power_28_min', 'reactive_power_28_std', 'power_29_avg', 'power_29_max', 'power_29_min', 'power_29_

In [0]:
# Make sure both event_id columns are the same type before joining
combined["event_id"] = combined["event_id"].astype(int)

# Merge the anomaly/normal label from event_info onto every row
combined = combined.merge(
    event_info[["event_id", "event_label", "event_description"]],
    on="event_id",
    how="left"
)

# Rename asset column in event_info to match sensor data naming
combined = combined.rename(columns={"asset": "asset_id_check"})
# This is just for reference — your join key is event_id so nothing breaks

# Quick check
print(combined["event_label"].value_counts())

event_label
anomaly    649789
normal     546958
Name: count, dtype: int64


In [0]:
display(combined.tail(10))

time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,wind_speed_3_min,wind_speed_3_std,sensor_5_avg,sensor_5_max,sensor_5_min,sensor_5_std,sensor_6_avg,sensor_7_avg,sensor_8_avg,sensor_9_avg,sensor_10_avg,sensor_11_avg,sensor_12_avg,sensor_13_avg,sensor_14_avg,sensor_15_avg,sensor_16_avg,sensor_17_avg,sensor_18_avg,sensor_18_max,sensor_18_min,sensor_18_std,sensor_19_avg,sensor_20_avg,sensor_21_avg,sensor_22_avg,sensor_23_avg,sensor_24_avg,sensor_25_avg,sensor_26_avg,reactive_power_27_avg,reactive_power_27_max,reactive_power_27_min,reactive_power_27_std,reactive_power_28_avg,reactive_power_28_max,reactive_power_28_min,reactive_power_28_std,power_29_avg,power_29_max,power_29_min,power_29_std,power_30_avg,power_30_max,power_30_min,power_30_std,sensor_31_avg,sensor_31_max,sensor_31_min,sensor_31_std,sensor_32_avg,sensor_33_avg,sensor_34_avg,sensor_35_avg,sensor_36_avg,sensor_37_avg,sensor_38_avg,sensor_39_avg,sensor_40_avg,sensor_41_avg,sensor_42_avg,sensor_43_avg,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg,event_id,event_label,event_description
2023-04-16 08:30:00,11,54057,prediction,0,15.0,98.7,-160.1,1.7000000000000002,1.7000000000000002,8.9,0.4,0.6000000000000001,24.0,24.1,24.0,0.0,25.0,32.0,27.0,32.0,27.0,32.0,35.0,25.0,24.0,30.0,30.0,29.0,3.7,16.1,0.0,5.7,18.0,25.0,29.0,1.0,2.5,3.4,4.4,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,-0.0017560975609756,-0.0113658536585365,0.0013170731707317,-3.414634146341E-4,-0.4,-16.6,2.0,402.7,401.2,399.1,29.0,29.0,29.0,62.0,65.0,60.0,28.0,258.8,24.0,-674.0,0.0,0.0,-123.0,0.0,0.0,-674.0,-123.0,0.0,0.0,0.0,0.0,18.0,92,normal,null
2023-04-16 08:40:00,11,54058,prediction,0,15.0,82.5,-176.3,2.2,2.2,8.1,0.5,0.6000000000000001,24.0,24.0,20.9,0.1,24.0,32.0,27.0,32.0,27.0,32.0,36.0,25.0,24.0,30.0,29.0,29.0,50.3,96.5,16.3,14.3,18.0,25.0,29.0,1.0,2.8,3.7,4.7,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0015121951219512,0.0296585365853658,0.0,0.0054146341463414,-0.0020975609756097,-0.0017560975609756,-0.0113658536585365,0.0014634146341463,-4.878048780487E-4,-0.4,-14.9,2.4,402.7,401.5,399.3,29.0,29.0,29.0,62.0,65.0,60.0,28.0,258.8,24.0,-706.0,0.0,0.0,-152.0,0.0,0.0,-706.0,-152.0,0.0,0.0,0.0,0.0,18.0,92,normal,null
2023-04-16 08:50:00,11,54059,prediction,0,15.0,106.3,-149.1,3.7,3.7,6.5,1.1,0.5,24.0,24.0,23.9,0.0,24.0,32.0,27.0,32.0,27.0,32.0,35.0,25.0,24.0,29.0,29.0,29.0,158.6,280.4,0.0,102.6,18.0,25.0,29.0,0.7000000000000001,8.9,7.8,9.4,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0286829268292682,0.0573170731707317,0.0087317073170731,0.0116097560975609,-0.0033170731707317,-0.0019024390243902,-0.0130731707317073,0.0014634146341463,-0.0038048780487804,-1.1,-24.7,4.6,403.4,402.3,400.3,29.0,29.0,29.0,62.0,65.0,60.0,28.0,255.4,24.0,-1141.0,0.0,0.0,-1304.0,0.0,0.0,-1141.0,-1304.0,1.1,2.5,0.0,1.2,17.0,92,normal,null
2023-04-16 09:00:00,11,54060,prediction,0,15.0,79.9,-21.2,3.3,3.3,6.9,1.2,0.6000000000000001,24.0,24.0,24.0,0.0,25.0,32.0,26.0,32.0,27.0,33.0,35.0,25.0,24.0,29.0,29.0,29.0,267.2,297.4,221.6,22.8,19.0,25.0,29.0,0.9,3.7,3.8,4.8,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0151219512195121,0.040780487804878,0.0,0.0099512195121951,-0.0022439024390243,-0.0019024390243902,-0.0116097560975609,0.0012682926829268,-8.780487804878E-4,-1.1,-19.4,2.6,402.2,400.9,398.8,29.0,29.0,29.0,62.0,65.0,60.0,28.0,101.1,23.0,-761.0,0.0,0.0,-290.0,0.0,0.0,-761.0,-290.0,2.4,2.6,2.0,0.2,18.0,92,normal,null
2023-04-16 09:10:00,11,54061,prediction,0,15.0,98.2,0.2,2.8,2.8,7.1,0.5,0.6000000000000001,24.0,24.0,21.0,0.1,24.0,32.0,26.0,32.0,27.0,34.0,35.0,25.0,24.0,29.0,29.0,29.0,216.1,271.9,175.1,27.4,19.0,25.0,29.0,0.9,4.1,4.2,5.1,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0030243902439024,0.0256585365853658,0.0,0.0060487804878048,-0.0023414634146341,-0.0019024390243902,-0.0133170731707317,0.001560975609756,-0.0010243902439024,-1.0,-27.7,3.4,401.0,

In [0]:
spark_df = spark.createDataFrame(combined)

(spark_df.write
    .format("delta")
    .mode("overwrite")
    #needs to be changed to your path
    .saveAsTable("finalcomp.default.farm_a_bronze")
)

print("Done. Row count:")
spark.sql("SELECT COUNT(*) FROM finalcomp.default.farm_a_bronze").show()

Done. Row count:
+--------+
|COUNT(*)|
+--------+
| 1196747|
+--------+



In [0]:
#load farm b
farm_b_path = f"{base_path}/Wind Farm B"

# Load event info
event_info_b = pd.read_csv(f"{farm_b_path}/event_info.csv", sep=";")

# Load all CSVs
datasets_path_b = f"{farm_b_path}/datasets"
csv_files_b = [f for f in os.listdir(datasets_path_b) if f.endswith(".csv")]
print(f"Farm B: found {len(csv_files_b)} files")

dfs_b = []
for fname in csv_files_b:
    df = pd.read_csv(os.path.join(datasets_path_b, fname), sep=";")
    df["event_id"] = int(fname.replace(".csv", ""))
    dfs_b.append(df)

combined_b = pd.concat(dfs_b, ignore_index=True)

# Join labels
combined_b["event_id"] = combined_b["event_id"].astype(int)
combined_b = combined_b.merge(
    event_info_b[["event_id", "event_label", "event_description"]],
    on="event_id",
    how="left"
)

print(f"Total rows: {len(combined_b):,}")
print(combined_b["event_label"].value_counts())

Farm B: found 15 files
Total rows: 859,065
event_label
normal     505688
anomaly    353377
Name: count, dtype: int64


In [0]:
temp_path = f"{base_path}/farm_b_temp.parquet"

combined_b.to_parquet(temp_path, index=False)

(spark.read.parquet(temp_path)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("finalcomp.default.farm_b_bronze")
)

# Clean up the temp file after writing
dbutils.fs.rm(temp_path)

print("Done. Row count:")
spark.sql("SELECT COUNT(*) FROM finalcomp.default.farm_b_bronze").show()

Done. Row count:
+--------+
|COUNT(*)|
+--------+
|  859065|
+--------+



In [0]:
spark.sql("SHOW TABLES IN finalcomp.default").show(truncate=False)

+--------+-------------+-----------+
|database|tableName    |isTemporary|
+--------+-------------+-----------+
|default |farm_a_bronze|false      |
|default |farm_b_bronze|false      |
+--------+-------------+-----------+



VERIFICATION

In [0]:
df_a = spark.table("finalcomp.default.farm_a_bronze")

print(f"Rows:    {df_a.count():,}")
print(f"Columns: {len(df_a.columns)}")

display(df_a.limit(5))

Rows:    1,196,747
Columns: 89


time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,wind_speed_3_min,wind_speed_3_std,sensor_5_avg,sensor_5_max,sensor_5_min,sensor_5_std,sensor_6_avg,sensor_7_avg,sensor_8_avg,sensor_9_avg,sensor_10_avg,sensor_11_avg,sensor_12_avg,sensor_13_avg,sensor_14_avg,sensor_15_avg,sensor_16_avg,sensor_17_avg,sensor_18_avg,sensor_18_max,sensor_18_min,sensor_18_std,sensor_19_avg,sensor_20_avg,sensor_21_avg,sensor_22_avg,sensor_23_avg,sensor_24_avg,sensor_25_avg,sensor_26_avg,reactive_power_27_avg,reactive_power_27_max,reactive_power_27_min,reactive_power_27_std,reactive_power_28_avg,reactive_power_28_max,reactive_power_28_min,reactive_power_28_std,power_29_avg,power_29_max,power_29_min,power_29_std,power_30_avg,power_30_max,power_30_min,power_30_std,sensor_31_avg,sensor_31_max,sensor_31_min,sensor_31_std,sensor_32_avg,sensor_33_avg,sensor_34_avg,sensor_35_avg,sensor_36_avg,sensor_37_avg,sensor_38_avg,sensor_39_avg,sensor_40_avg,sensor_41_avg,sensor_42_avg,sensor_43_avg,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg,event_id,event_label,event_description
2023-03-05 06:40:00,11,26333,train,0,13.0,72.5,164.9,1.5,1.5,6.9,0.4,0.8,24.0,24.0,23.9,0.0,24.0,31.0,26.0,30.0,25.0,36.0,38.0,24.0,24.0,30.0,30.0,29.0,0.0,0.0,0.0,0.0,16.0,23.0,29.0,1.0,2.7,3.6,4.4,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,-0.0017073170731707,-0.011170731707317,0.0012195121951219,-3.414634146341E-4,-0.3,-17.0,1.9,400.9,400.2,396.5,29.0,29.0,29.0,54.0,60.0,58.0,26.0,267.6,22.0,-690.0,0.0,0.0,-120.0,0.0,0.0,-690.0,-120.0,0.0,0.0,0.0,0.0,15.0,69,normal,null
2023-03-05 06:50:00,11,26334,train,0,13.0,139.9,-127.7,1.6,1.6,8.3,0.4,0.8,24.0,24.1,23.9,0.0,25.0,31.0,26.0,30.0,25.0,35.0,38.0,24.0,24.0,30.0,29.0,29.0,0.0,0.0,0.0,0.0,16.0,23.0,29.0,1.0,2.4,3.4,4.3,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0019512195121951,-0.0017073170731707,-0.0114634146341463,0.001170731707317,-3.414634146341E-4,-0.3,-18.4,1.9,401.7,400.9,397.1,29.0,29.0,29.0,54.0,60.0,58.0,26.0,267.6,23.0,-662.0,0.0,0.0,-112.0,0.0,0.0,-662.0,-112.0,0.0,0.0,0.0,0.0,15.0,69,normal,null
2023-03-05 07:00:00,11,26335,train,0,13.0,208.4,-59.1,1.8,1.8,8.3,0.4,0.7000000000000001,24.0,24.1,20.8,0.1,25.0,31.0,25.0,30.0,25.0,35.0,38.0,24.0,23.0,29.0,29.0,29.0,0.0,0.0,0.0,0.0,16.0,23.0,29.0,1.0,2.5,3.5,4.4,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,-0.0017073170731707,-0.0114634146341463,0.001170731707317,-3.414634146341E-4,-0.4,-18.1,1.8,402.4,401.8,398.1,29.0,29.0,29.0,54.0,60.0,58.0,26.0,267.6,23.0,-681.0,0.0,0.0,-119.0,0.0,0.0,-681.0,-119.0,0.0,0.0,0.0,0.0,15.0,69,normal,null
2023-03-05 07:10:00,11,26336,train,0,13.0,172.4,-95.2,2.6,2.6,10.8,0.4,1.0,24.0,24.1,23.9,0.0,24.0,31.0,25.0,30.0,25.0,35.0,38.0,24.0,23.0,29.0,29.0,29.0,24.1,71.2,0.0,28.8,16.0,23.0,29.0,1.0,2.6,3.6,4.4,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.756097560975E-4,0.0246829268292682,0.0,0.0030731707317073,-0.002,-0.0017073170731707,-0.0114634146341463,0.0012195121951219,-3.414634146341E-4,-0.3,-16.8,1.9,399.7,398.8,395.1,29.0,29.0,29.0,54.0,60.0,58.0,26.0,267.6,23.0,-687.0,0.0,0.0,-123.0,0.0,0.0,-687.0,-123.0,0.0,0.0,0.0,0.0,15.0,69,normal,null
2023-03-05 07:20:00,11,26337,train,0,13.0,218.0,-49.5,2.5,2.5,14.0,0.7000000000000001,0.7000000000000001,24.0,24.0,23.9,0.0,23.0,31.0,25.0,30.0,25.0,34.0,39.0,24.0,23.0,29.0,29.0,29.0,69.6,106.3,41.4,18.8,16.0,22.0,29.0,1.0,2.6,3.5,4.3,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.439024390243E-4,0.0190731707317073,0.0,0.0015121951219512,-0.0019512195121951,-0.0017073170731707,-0.0112682926829268,9.756097560975E-4,-3.414634146341E-4,-0.4,-13.0,1.6,400.3,399.5,395.9,29.0,29.0,29.0,54.0,60.0,58.0,26.0,267.6,23.0,-676.0,0.0,0.0,-114.0,0.0,0.0,-676.0,-114.0,0.0,0.0,0.0,0.0,15.0,69,normal,null


In [0]:
# Every row should have a label — no nulls here means the join worked
df_a.groupBy("event_label").count().show()

# Should show something like:
# anomaly | 44 events worth of rows
# normal  | 51 events worth of rows

+-----------+------+
|event_label| count|
+-----------+------+
|     normal|546958|
|    anomaly|649789|
+-----------+------+



In [0]:
df_a.groupBy("train_test").count().show()
# Should show "train" and "test" (or similar strings)

+----------+-------+
|train_test|  count|
+----------+-------+
|     train|1146154|
|prediction|  50593|
+----------+-------+



In [0]:
from pyspark.sql import functions as F

key_cols = ["time_stamp", "asset_id", "event_id", "event_label", "train_test"]

df_a.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in key_cols
]).show()
# All should be 0

+----------+--------+--------+-----------+----------+
|time_stamp|asset_id|event_id|event_label|train_test|
+----------+--------+--------+-----------+----------+
|         0|       0|       0|          0|         0|
+----------+--------+--------+-----------+----------+



In [0]:
# Farm A should have 22 datasets total
df_a.select("event_id").distinct().count()

22

In [0]:
df_b = spark.table("finalcomp.default.farm_b_bronze")

print(f"Rows:    {df_b.count():,}")
print(f"Columns: {len(df_b.columns)}")

display(df_b.limit(5))

Rows:    859,065
Columns: 260


time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_0_max,sensor_0_min,sensor_0_std,sensor_1_avg,sensor_1_max,sensor_1_min,sensor_1_std,sensor_2_avg,sensor_2_max,sensor_2_min,sensor_2_std,sensor_3_avg,sensor_3_max,sensor_3_min,sensor_3_std,sensor_4_avg,sensor_4_max,sensor_4_min,sensor_4_std,sensor_5_avg,sensor_5_max,sensor_5_min,sensor_5_std,sensor_6_avg,sensor_6_max,sensor_6_min,sensor_6_std,sensor_7_avg,sensor_7_max,sensor_7_min,sensor_7_std,sensor_8_avg,sensor_8_max,sensor_8_min,sensor_8_std,sensor_9_avg,sensor_9_max,sensor_9_min,sensor_9_std,sensor_10_avg,sensor_10_max,sensor_10_min,sensor_10_std,reactive_power_11_avg,reactive_power_11_max,reactive_power_11_min,reactive_power_11_std,sensor_12_avg,sensor_12_max,sensor_12_min,sensor_12_std,sensor_13_avg,sensor_13_max,sensor_13_min,sensor_13_std,sensor_14_avg,sensor_14_max,sensor_14_min,sensor_14_std,sensor_15_avg,sensor_15_max,sensor_15_min,sensor_15_std,sensor_16_avg,sensor_16_max,sensor_16_min,sensor_16_std,sensor_17_avg,sensor_17_max,sensor_17_min,sensor_17_std,sensor_18_avg,sensor_18_max,sensor_18_min,sensor_18_std,sensor_19_avg,sensor_19_max,sensor_19_min,sensor_19_std,sensor_20_avg,sensor_20_max,sensor_20_min,sensor_20_std,sensor_21_avg,sensor_21_max,sensor_21_min,sensor_21_std,sensor_22_avg,sensor_22_max,sensor_22_min,sensor_22_std,sensor_23_avg,sensor_23_max,sensor_23_min,sensor_23_std,sensor_24_avg,sensor_24_max,sensor_24_min,sensor_24_std,sensor_25_avg,sensor_25_max,sensor_25_min,sensor_25_std,sensor_26_avg,sensor_26_max,sensor_26_min,sensor_26_std,sensor_27_avg,sensor_27_max,sensor_27_min,sensor_27_std,sensor_28_avg,sensor_28_max,sensor_28_min,sensor_28_std,sensor_29_avg,sensor_29_max,sensor_29_min,sensor_29_std,sensor_30_avg,sensor_30_max,sensor_30_min,sensor_30_std,sensor_31_avg,sensor_31_max,sensor_31_min,sensor_31_std,sensor_32_avg,sensor_32_max,sensor_32_min,sensor_32_std,sensor_33_avg,sensor_33_max,sensor_33_min,sensor_33_std,sensor_34_avg,sensor_34_max,sensor_34_min,sensor_34_std,sensor_35_avg,sensor_35_max,sensor_35_min,sensor_35_std,sensor_36_avg,sensor_36_max,sensor_36_min,sensor_36_std,sensor_37_avg,sensor_37_max,sensor_37_min,sensor_37_std,sensor_38_avg,sensor_38_max,sensor_38_min,sensor_38_std,sensor_39_avg,sensor_39_max,sensor_39_min,sensor_39_std,sensor_40_avg,sensor_40_max,sensor_40_min,sensor_40_std,sensor_41_avg,sensor_41_max,sensor_41_min,sensor_41_std,sensor_42_avg,sensor_42_max,sensor_42_min,sensor_42_std,sensor_43_avg,sensor_43_max,sensor_43_min,sensor_43_std,sensor_44_avg,sensor_44_max,sensor_44_min,sensor_44_std,sensor_45_avg,sensor_45_max,sensor_45_min,sensor_45_std,sensor_46_avg,sensor_46_max,sensor_46_min,sensor_46_std,sensor_47_avg,sensor_47_max,sensor_47_min,sensor_47_std,sensor_48_avg,sensor_48_max,sensor_48_min,sensor_48_std,sensor_49_avg,sensor_49_max,sensor_49_min,sensor_49_std,sensor_50_avg,sensor_50_max,sensor_50_min,sensor_50_std,sensor_51_avg,sensor_51_max,sensor_51_min,sensor_51_std,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg,sensor_53_max,sensor_53_min,sensor_53_std,sensor_54_avg,sensor_54_max,sensor_54_min,sensor_54_std,sensor_55_avg,sensor_55_max,sensor_55_min,sensor_55_std,sensor_56_avg,sensor_56_max,sensor_56_min,sensor_56_std,sensor_57_avg,sensor_57_max,sensor_57_min,sensor_57_std,power_58_avg,power_58_max,power_58_min,power_58_std,wind_speed_59_avg,wind_speed_59_max,wind_speed_59_min,wind_speed_59_std,wind_speed_60_avg,wind_speed_60_max,wind_speed_60_min,wind_speed_60_std,wind_speed_61_avg,wind_speed_61_max,wind_speed_61_min,wind_speed_61_std,power_62_avg,power_62_max,power_62_min,power_62_std,event_id,event_label,event_description
2022-02-02 00:00:00,11,0,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.0,0.0,0.0,0.0,766.0,0.0,0.0,0.0,303.5,327.01,5.13,295.03,-2161.98,0.0,0.0,0.0,17.8,24.21,2.98,6.04,15.83,20.44,3.59,11.1,7.79,7.9,0.08,7.7,217.39,0.0,0.0,0.0,21.4,30.65,3.71,17.44,0.0332887096774193,0.0554725806451612,0.0176774193548387,0.0024145161290322,37846.1,53386.1,13595.1,1385

In [0]:
# Every row should have a label — no nulls here means the join worked
df_b.groupBy("event_label").count().show()

# Should show something like:
# anomaly | 44 events worth of rows
# normal  | 51 events worth of rows

+-----------+------+
|event_label| count|
+-----------+------+
|     normal|505688|
|    anomaly|353377|
+-----------+------+



In [0]:
df_b.groupBy("train_test").count().show()
# Should show "train" and "test" (or similar strings)

+----------+------+
|train_test| count|
+----------+------+
|prediction| 72128|
|     train|786937|
+----------+------+



In [0]:
from pyspark.sql import functions as F

key_cols = ["time_stamp", "asset_id", "event_id", "event_label", "train_test"]

df_b.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in key_cols
]).show()
# All should be 0

+----------+--------+--------+-----------+----------+
|time_stamp|asset_id|event_id|event_label|train_test|
+----------+--------+--------+-----------+----------+
|         0|       0|       0|          0|         0|
+----------+--------+--------+-----------+----------+



In [0]:
# Farm B should have 15 datasets total
df_b.select("event_id").distinct().count()

15